**Cell 1**

# NB11.1 — Multi-seed TinyLlama HelpSteer2 DPO experts

This notebook measures the seed stability of the five NB11 DPO expert
directions. The preference-pair subset is fixed once with
`PAIR_SEED=137`; only the training seed varies over
`{137, 138, 139, 140, 141}`. Consequently, every seed uses exactly the
same 2,690 pairs per attribute, while LoRA initialization, minibatch
order, and other training randomness may vary.

For every seed, five experts are trained and two relationship matrices
are saved: the effective-update Gram matrix and its cosine-normalized
version. `R_seed137`, ..., `R_seed141` refer to the cosine matrices.
Their elementwise mean, sample standard deviation, eigenvalues, and
distances from the mean are reported descriptively.

Finally, the five seed runs for each attribute are averaged in effective
update space,

$$\bar\Delta_i=\frac{1}{5}\sum_{s=137}^{141}\Delta_{i,s}.$$

Averages of the LoRA factors themselves are mathematically invalid.
Instead, NB11.1 concatenates the factors and saves each arithmetic mean
exactly as a rank-40 LoRA adapter. The matrix computed from these five
mean adapters is reported separately as `R_mean_adapters`; it need not
equal the elementwise matrix mean `R_bar`.

**Run all is supported.** Google Drive is mounted first. Completed jobs
are restored and verified before being skipped. If the original verified
NB11 seed-137 run exists on Drive, it is reused; otherwise seed 137 is
trained again. ArmoRM is never loaded in this notebook.


**Cell 2**

## 1. Mount Drive, then clone or update the repository


In [ ]:
# Cell 3
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

%cd /content
import os, shutil
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis


**Cell 4**

## 2. Check the GPU

One bf16-capable GPU is required. The 25 jobs run sequentially, so they
do not require more VRAM than one ordinary NB11 training job.


In [ ]:
# Cell 5
!nvidia-smi


**Cell 6**

## 3. Install the pinned NB11 dependencies

The original tested TRL/Transformers combination is retained. No reward
model package is required.


In [ ]:
# Cell 7
import importlib.metadata
import subprocess
import sys

print(f"Python {sys.version.split()[0]} ({sys.executable})")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
packages = [
    "pandas==2.2.3",
    "numpy==2.1.3",
    "protobuf==5.29.5",
    "transformers==4.45.2",
    "tokenizers==0.20.3",
    "peft==0.13.2",
    "accelerate==1.1.1",
    "trl==0.11.4",
    "datasets==3.1.0",
    "huggingface_hub==0.36.0",
    "bitsandbytes",
    "pyyaml",
    "safetensors",
]
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "--only-binary=:all:",
        *packages,
    ],
    check=True,
)

expected = {"transformers": "4.45.2", "trl": "0.11.4"}
installed = {name: importlib.metadata.version(name) for name in expected}
assert installed == expected, (
    f"Incompatible DPO stack: {installed}; expected {expected}. "
    "Restart the runtime and run all again."
)

import accelerate, peft, tokenizers, transformers, trl
print(
    "Runtime:", sys.version.split()[0],
    "transformers", transformers.__version__,
    "tokenizers", tokenizers.__version__,
    "peft", peft.__version__,
    "accelerate", accelerate.__version__,
    "trl", trl.__version__,
)


**Cell 8**

## 4. Frozen settings and persistent paths

The exact base and dataset revisions from NB11/NB13 are hard-coded. The
pair seed remains fixed at 137; `TRAIN_SEEDS` is the only seed grid.


In [ ]:
# Cell 9
from pathlib import Path
import json, os, shutil

PROJECT_ROOT = Path("/content/master-thesis").resolve()
RUN_TAG = "nb11_1_multiseed_pair137_train137_141"
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
BASE_REVISION = "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
DATASET_NAME = "nvidia/HelpSteer2"
DATASET_REVISION = "990b2711a36180dd19d9c94b8627844866f8982a"
ATTRIBUTES = ["helpfulness", "correctness", "coherence", "complexity", "verbosity"]
PAIR_SEED = 137
TRAIN_SEEDS = (137, 138, 139, 140, 141)

MAX_PAIRS = 2690
DPO_BETA = 0.1
EPOCHS = 1.0
LEARNING_RATE = 5e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256
RUN_SMOKE_TEST = True
REUSE_VERIFIED_NB11_SEED137 = True
DELETE_COMPLETED_TRAINER_STATE = True

OUTPUT_ROOT = PROJECT_ROOT / "results" / "dpo_rq2" / RUN_TAG
ANALYSIS_ROOT = OUTPUT_ROOT / "geometry"
MEAN_ROOT = OUTPUT_ROOT / "mean_adapters"
DRIVE_ROOT = Path("/content/drive/MyDrive/master-thesis-nb11-1") / RUN_TAG
DRIVE_TRAINED_ROOT = DRIVE_ROOT / "trained"
DRIVE_ANALYSIS_ROOT = DRIVE_ROOT / "geometry"
DRIVE_MEAN_ROOT = DRIVE_ROOT / "mean_adapters"
LEGACY_SEED137_ROOT = (
    Path("/content/drive/MyDrive/master-thesis-nb11")
    / "nb11_pairs2690_seed137_run1"
)
for directory in (
    OUTPUT_ROOT, ANALYSIS_ROOT, MEAN_ROOT, DRIVE_ROOT,
    DRIVE_TRAINED_ROOT, DRIVE_ANALYSIS_ROOT, DRIVE_MEAN_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

def seed_root(seed):
    return OUTPUT_ROOT / f"seed_{int(seed)}"

def drive_seed_root(seed):
    return DRIVE_TRAINED_ROOT / f"seed_{int(seed)}"

print(f"run tag            = {RUN_TAG}")
print(f"pair seed          = {PAIR_SEED} (fixed)")
print(f"training seeds     = {TRAIN_SEEDS}")
print(f"training jobs      = {len(TRAIN_SEEDS) * len(ATTRIBUTES)}")
print(f"local output       = {OUTPUT_ROOT}")
print(f"persistent output  = {DRIVE_ROOT}")
print(f"base revision      = {BASE_REVISION}")
print(f"dataset revision   = {DATASET_REVISION}")


**Cell 10**

## 5. Validate the central experiment configuration


In [ ]:
# Cell 11
!python scripts/validate_tinyllama_helpsteer2_config.py
from src.experiment_config import get_attribute_order, load_experiment_config

cfg = load_experiment_config(PROJECT_ROOT / "configs/tinyllama_helpsteer2_armorm.yaml")
assert list(get_attribute_order(cfg)) == ATTRIBUTES
assert cfg["base_model_name"] == BASE_MODEL
assert cfg["dataset_name"] == DATASET_NAME
assert PAIR_SEED not in set(TRAIN_SEEDS[1:])
print("[OK] Central axis order and frozen model/dataset identities match.")


**Cell 12**

## 6. Freeze the pair subsets and restore verified work

Each attribute's selected-pair hash is computed once with seed 137 and
checked against every completed expert. A verified original NB11 seed-137
expert may be imported to avoid repeating identical work.


In [ ]:
# Cell 13
import hashlib
import importlib.util
import tempfile
from datasets import load_dataset

spec = importlib.util.spec_from_file_location(
    "nb11_dpo_script", PROJECT_ROOT / "scripts/0_dpo_expert.py"
)
dpo_script = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(dpo_script)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

train_rows = load_dataset(
    DATASET_NAME, split="train", revision=DATASET_REVISION
)
PAIR_ID_HASHES = {}
for axis in ATTRIBUTES:
    candidates = dpo_script.build_pairs_from_rows(train_rows, axis)
    selected = dpo_script.select_pairs(
        candidates, seed=PAIR_SEED, max_pairs=MAX_PAIRS
    )
    pair_hash = dpo_script.canonical_hash(
        {"pair_ids": [pair["pair_id"] for pair in selected]}
    )
    PAIR_ID_HASHES[axis] = pair_hash
    print(
        f"{axis:12} candidates={len(candidates):5d} "
        f"selected={len(selected):4d} pairs_sha256={pair_hash[:16]}..."
    )
del train_rows

EXPECTED_LORA = {
    "r": 8,
    "alpha": 16,
    "dropout": 0.05,
    "bias": "none",
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
}

def expert_dir(root, axis):
    return Path(root) / f"dpo_{axis}"

def verify_expert(root, seed, axis):
    directory = expert_dir(root, axis)
    manifest_path = directory / "training_manifest.json"
    weights_path = directory / "adapter/adapter_model.safetensors"
    if not manifest_path.is_file() or not weights_path.is_file():
        raise FileNotFoundError(f"Incomplete expert: {directory}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = {
        "completed": True,
        "training_method": "DPO",
        "reward_name": axis,
        "base_model_name": BASE_MODEL,
        "base_revision": BASE_REVISION,
        "dataset_name": DATASET_NAME,
        "dataset_revision": DATASET_REVISION,
        "dataset_split": "train",
        "pair_seed": PAIR_SEED,
        "selected_pair_count": MAX_PAIRS,
        "selected_pair_ids_sha256": PAIR_ID_HASHES[axis],
        "beta": DPO_BETA,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM,
        "max_length": MAX_LENGTH,
        "max_prompt_length": MAX_PROMPT_LENGTH,
        "precision": "bf16",
        "seed": int(seed),
        "armorm_used_during_training": False,
        "checkpoint_selection": "fixed final epoch; no reward-model selection",
    }
    for field, value in expected.items():
        if manifest.get(field) != value:
            raise RuntimeError(
                f"{directory} differs on {field}: "
                f"{manifest.get(field)!r} != {value!r}"
            )
    if manifest.get("lora") != EXPECTED_LORA:
        raise RuntimeError(f"{directory} has a different LoRA configuration")
    digest = sha256_file(weights_path)
    if digest != manifest.get("adapter_model_sha256"):
        raise RuntimeError(f"Adapter hash mismatch: {directory}")
    return manifest

def copy_compact_expert(source, destination):
    source, destination = Path(source), Path(destination)
    if destination.exists():
        raise FileExistsError(destination)
    destination.mkdir(parents=True)
    shutil.copytree(source / "adapter", destination / "adapter")
    for filename in (
        "training_manifest.json", "run_binding.json", "source_provenance.json"
    ):
        candidate = source / filename
        if candidate.is_file():
            shutil.copy2(candidate, destination / filename)

def publish_compact_expert(source, destination, seed, axis):
    source, destination = Path(source), Path(destination)
    if destination.exists():
        source_hash = verify_expert(source.parent, seed, axis)["adapter_model_sha256"]
        saved_hash = verify_expert(destination.parent, seed, axis)["adapter_model_sha256"]
        if source_hash != saved_hash:
            raise RuntimeError(f"Drive copy differs: {destination}")
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_parent = Path(
        tempfile.mkdtemp(prefix=f".{destination.name}.", dir=str(destination.parent))
    )
    temporary = temporary_parent / destination.name
    try:
        copy_compact_expert(source, temporary)
        verify_expert(temporary.parent, seed, axis)
        os.replace(temporary, destination)
    finally:
        shutil.rmtree(temporary_parent, ignore_errors=True)

# Restore NB11.1 work first; all restored files are verified below.
for seed in TRAIN_SEEDS:
    for axis in ATTRIBUTES:
        local = expert_dir(seed_root(seed), axis)
        saved = expert_dir(drive_seed_root(seed), axis)
        if not local.exists() and saved.exists():
            copy_compact_expert(saved, local)

# Import the exact original seed-137 adapters when available.
if REUSE_VERIFIED_NB11_SEED137:
    for axis in ATTRIBUTES:
        local = expert_dir(seed_root(137), axis)
        legacy = expert_dir(LEGACY_SEED137_ROOT, axis)
        if not local.exists() and legacy.exists():
            verify_expert(LEGACY_SEED137_ROOT, 137, axis)
            copy_compact_expert(legacy, local)
            provenance = {
                "source": "verified original NB11 seed-137 run",
                "source_path": str(legacy),
                "adapter_model_sha256": verify_expert(seed_root(137), 137, axis)[
                    "adapter_model_sha256"
                ],
            }
            (local / "source_provenance.json").write_text(
                json.dumps(provenance, indent=2, sort_keys=True) + "\n",
                encoding="utf-8",
            )
            publish_compact_expert(
                local, expert_dir(drive_seed_root(137), axis), 137, axis
            )
            print(f"[reuse] verified original seed 137 / {axis}")

existing_jobs = 0
for seed in TRAIN_SEEDS:
    for axis in ATTRIBUTES:
        local = expert_dir(seed_root(seed), axis)
        weights = local / "adapter/adapter_model.safetensors"
        manifest = local / "training_manifest.json"
        if weights.is_file() and manifest.is_file():
            verify_expert(seed_root(seed), seed, axis)
            existing_jobs += 1
        elif local.exists():
            print(f"[resume-ready] local partial job: seed {seed} / {axis}")
print(f"[OK] Fixed pair subsets. Verified existing jobs: {existing_jobs}/25")


**Cell 14**

## 7. Evaluator-free smoke test

This runs only when at least one of the 25 real jobs is missing.


In [ ]:
# Cell 15
import subprocess, sys
from collections import deque

def run_logged_subprocess(command, label):
    print(f"[run] {label}: {' '.join(map(str, command))}", flush=True)
    tail = deque(maxlen=80)
    process = subprocess.Popen(
        command,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip())
    return_code = process.wait()
    if return_code:
        final_output = "\n".join(tail)
        raise RuntimeError(
            f"{label} failed with exit code {return_code}.\n"
            f"Last subprocess output:\n{final_output}"
        )

def is_complete(seed, axis):
    directory = expert_dir(seed_root(seed), axis)
    weights = directory / "adapter/adapter_model.safetensors"
    manifest = directory / "training_manifest.json"
    if not weights.is_file() or not manifest.is_file():
        return False
    verify_expert(seed_root(seed), seed, axis)
    return True

all_jobs_complete = all(
    is_complete(seed, axis)
    for seed in TRAIN_SEEDS
    for axis in ATTRIBUTES
)
smoke_root = Path("/tmp/nb11_1_dpo_smoke")
if RUN_SMOKE_TEST and not all_jobs_complete:
    shutil.rmtree(smoke_root, ignore_errors=True)
    command = [
        sys.executable, "-u", "scripts/0_dpo_expert.py",
        "--reward_name", "helpfulness",
        "--base_model_name", BASE_MODEL,
        "--base_revision", BASE_REVISION,
        "--dataset_name", DATASET_NAME,
        "--dataset_revision", DATASET_REVISION,
        "--output_root", str(smoke_root),
        "--max_pairs", "16",
        "--pair_seed", str(PAIR_SEED),
        "--seed", "137",
        "--epochs", "1",
        "--save_steps", "1000",
        "--overwrite",
    ]
    run_logged_subprocess(command, "DPO smoke test")
    shutil.rmtree(smoke_root, ignore_errors=True)
    print("[OK] Smoke test passed and was deleted.")
elif all_jobs_complete:
    print("[skip] All 25 verified jobs already exist.")
else:
    print("Smoke test disabled.")


**Cell 16**

## 8. Multi-seed training helper

Each completed expert is verified, compactly published to Drive, and
skipped on exact reruns. Trainer checkpoints are removed only after the
final adapter has been verified and backed up.


In [ ]:
# Cell 17
import gc, torch

def train_axis(seed, axis):
    seed = int(seed)
    assert seed in TRAIN_SEEDS and axis in ATTRIBUTES
    root = seed_root(seed)
    directory = expert_dir(root, axis)
    weights = directory / "adapter/adapter_model.safetensors"
    manifest = directory / "training_manifest.json"
    if weights.is_file() and manifest.is_file():
        verify_expert(root, seed, axis)
        print(f"[skip] verified seed {seed} / {axis}")
        return

    command = [
        sys.executable, "-u", "scripts/0_dpo_expert.py",
        "--reward_name", axis,
        "--base_model_name", BASE_MODEL,
        "--base_revision", BASE_REVISION,
        "--dataset_name", DATASET_NAME,
        "--dataset_revision", DATASET_REVISION,
        "--split", "train",
        "--output_root", str(root),
        "--beta", str(DPO_BETA),
        "--epochs", str(EPOCHS),
        "--lr", str(LEARNING_RATE),
        "--batch_size", str(BATCH_SIZE),
        "--grad_accum", str(GRAD_ACCUM),
        "--max_length", str(MAX_LENGTH),
        "--max_prompt_length", str(MAX_PROMPT_LENGTH),
        "--max_pairs", str(MAX_PAIRS),
        "--pair_seed", str(PAIR_SEED),
        "--seed", str(seed),
        "--resume",
    ]
    print(f"\n=== DPO seed {seed} / {axis} ===")
    run_logged_subprocess(command, f"DPO training (seed {seed}, {axis})")
    verify_expert(root, seed, axis)
    publish_compact_expert(
        directory, expert_dir(drive_seed_root(seed), axis), seed, axis
    )
    print(f"[backup] seed {seed} / {axis} -> Drive")
    if DELETE_COMPLETED_TRAINER_STATE:
        trainer_dir = directory / "trainer"
        if trainer_dir.exists():
            shutil.rmtree(trainer_dir)
            print(f"[cleanup] removed completed trainer state for seed {seed} / {axis}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


**Cell 18**

## 9. Train all missing seed/attribute combinations

This is the long-running cell. Re-running it is safe: verified completed
jobs are skipped, and a compatible local partial job is resumed.


In [ ]:
# Cell 19
total_jobs = len(TRAIN_SEEDS) * len(ATTRIBUTES)
job_number = 0
for seed in TRAIN_SEEDS:
    for axis in ATTRIBUTES:
        job_number += 1
        print(f"\n##### Job {job_number}/{total_jobs}: seed={seed}, axis={axis} #####")
        train_axis(seed, axis)
print("[OK] All 25 DPO experts are complete.")


**Cell 20**

## 10. Verify the complete 25-adapter inventory

Pair hashes must be identical across training seeds for each attribute;
adapter hashes should normally differ.


In [ ]:
# Cell 21
import pandas as pd

inventory_rows = []
for seed in TRAIN_SEEDS:
    for axis in ATTRIBUTES:
        root = seed_root(seed)
        manifest = verify_expert(root, seed, axis)
        drive_manifest = verify_expert(drive_seed_root(seed), seed, axis)
        assert drive_manifest["adapter_model_sha256"] == manifest["adapter_model_sha256"]
        provenance_path = expert_dir(root, axis) / "source_provenance.json"
        provenance = (
            json.loads(provenance_path.read_text(encoding="utf-8"))["source"]
            if provenance_path.is_file()
            else "NB11.1 training or restore"
        )
        inventory_rows.append({
            "seed": seed,
            "attribute": axis,
            "pair_seed": manifest["pair_seed"],
            "selected_pair_ids_sha256": manifest["selected_pair_ids_sha256"],
            "adapter_model_sha256": manifest["adapter_model_sha256"],
            "trainer_script_sha256": manifest["trainer_script_sha256"],
            "source": provenance,
            "drive_path": str(expert_dir(drive_seed_root(seed), axis)),
        })

inventory_df = pd.DataFrame(inventory_rows)
assert len(inventory_df) == 25
assert (inventory_df["pair_seed"] == PAIR_SEED).all()
assert (inventory_df.groupby("attribute")["selected_pair_ids_sha256"].nunique() == 1).all()
INVENTORY_CSV = ANALYSIS_ROOT / "adapter_inventory.csv"
inventory_df.to_csv(INVENTORY_CSV, index=False)
display(inventory_df)
print("[OK] 25/25 local and Drive adapters verified; pair subset is fixed within every attribute.")


**Cell 22**

## 11. Compute and compare $R_{137},\ldots,R_{141}$

The primary matrices shown as `R_seed...` are cosine similarities of the
complete effective LoRA updates. Unnormalized Gram matrices are saved as
well. `R_bar` is the elementwise mean across the five cosine matrices.


In [ ]:
# Cell 23
import numpy as np
from src.effective_lora_geometry import (
    effective_lora_inner_product,
    effective_lora_update_norm,
    load_effective_lora_geometry,
    validate_compatible_geometries,
)

def relationship_matrices(adapter_paths):
    geometries = {
        axis: load_effective_lora_geometry(adapter_paths[axis])
        for axis in ATTRIBUTES
    }
    validate_compatible_geometries(
        [geometries[axis] for axis in ATTRIBUTES], ATTRIBUTES
    )
    gram = np.empty((len(ATTRIBUTES), len(ATTRIBUTES)), dtype=np.float64)
    for i, left in enumerate(ATTRIBUTES):
        for j, right in enumerate(ATTRIBUTES[: i + 1]):
            value = effective_lora_inner_product(
                geometries[left], geometries[right]
            )
            gram[i, j] = gram[j, i] = value
    norms = np.asarray([
        effective_lora_update_norm(geometries[axis])
        for axis in ATTRIBUTES
    ])
    cosine = gram / np.outer(norms, norms)
    cosine = 0.5 * (cosine + cosine.T)
    if not np.allclose(np.diag(cosine), 1.0, atol=1e-10):
        raise RuntimeError("Cosine matrix has a non-unit diagonal")
    return gram, cosine, norms

R_GRAM_BY_SEED, R_COS_BY_SEED = {}, {}
geometry_rows = []
for seed in TRAIN_SEEDS:
    paths = {
        axis: expert_dir(seed_root(seed), axis) / "adapter"
        for axis in ATTRIBUTES
    }
    gram, cosine, norms = relationship_matrices(paths)
    R_GRAM_BY_SEED[seed] = gram
    R_COS_BY_SEED[seed] = cosine
    globals()[f"R_seed{seed}"] = cosine
    globals()[f"R_gram_seed{seed}"] = gram
    pd.DataFrame(gram, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(
        ANALYSIS_ROOT / f"R_seed{seed}_gram.csv"
    )
    pd.DataFrame(cosine, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(
        ANALYSIS_ROOT / f"R_seed{seed}_cos.csv"
    )
    eigenvalues = np.linalg.eigvalsh(cosine)
    offdiag = cosine[np.triu_indices(len(ATTRIBUTES), 1)]
    geometry_rows.append({
        "seed": seed,
        "min_eigenvalue": float(eigenvalues.min()),
        "max_eigenvalue": float(eigenvalues.max()),
        "condition_number": (
            float(eigenvalues.max() / eigenvalues.min())
            if eigenvalues.min() > 1e-12 else float("inf")
        ),
        "offdiag_min": float(offdiag.min()),
        "offdiag_max": float(offdiag.max()),
        "offdiag_mean": float(offdiag.mean()),
        **{f"norm_{axis}": float(norm) for axis, norm in zip(ATTRIBUTES, norms)},
    })
    print(f"\nR_seed{seed} (effective-update cosine)")
    display(pd.DataFrame(cosine, index=ATTRIBUTES, columns=ATTRIBUTES).round(5))

cosine_stack = np.stack([R_COS_BY_SEED[seed] for seed in TRAIN_SEEDS])
gram_stack = np.stack([R_GRAM_BY_SEED[seed] for seed in TRAIN_SEEDS])
R_bar = cosine_stack.mean(axis=0)
R_std = cosine_stack.std(axis=0, ddof=1)
R_gram_bar = gram_stack.mean(axis=0)
R_gram_std = gram_stack.std(axis=0, ddof=1)
pd.DataFrame(R_bar, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(
    ANALYSIS_ROOT / "R_seed_mean_cos.csv"
)
pd.DataFrame(R_std, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(
    ANALYSIS_ROOT / "R_seed_sample_std_cos.csv"
)
pd.DataFrame(R_gram_bar, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(
    ANALYSIS_ROOT / "R_seed_mean_gram.csv"
)
pd.DataFrame(R_gram_std, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(
    ANALYSIS_ROOT / "R_seed_sample_std_gram.csv"
)

geometry_df = pd.DataFrame(geometry_rows)
geometry_df["frobenius_distance_to_R_bar"] = [
    float(np.linalg.norm(R_COS_BY_SEED[seed] - R_bar, ord="fro"))
    for seed in TRAIN_SEEDS
]
GEOMETRY_SUMMARY_CSV = ANALYSIS_ROOT / "seed_geometry_summary.csv"
geometry_df.to_csv(GEOMETRY_SUMMARY_CSV, index=False)
print("\nR_bar: elementwise mean of the five R_seed cosine matrices")
display(pd.DataFrame(R_bar, index=ATTRIBUTES, columns=ATTRIBUTES).round(5))
print("\nSample standard deviation across seeds")
display(pd.DataFrame(R_std, index=ATTRIBUTES, columns=ATTRIBUTES).round(5))
display(geometry_df.round(6))

geometry_report = {
    "schema_version": 1,
    "pair_seed": PAIR_SEED,
    "training_seeds": list(TRAIN_SEEDS),
    "attribute_order": ATTRIBUTES,
    "primary_R": "cosine similarity of complete effective LoRA updates",
    "R_seed_mean_role": "elementwise descriptive mean across seed matrices",
    "R_seed_sample_std_ddof": 1,
    "per_seed": geometry_rows,
    "frobenius_distance_to_R_bar": dict(zip(
        map(str, TRAIN_SEEDS), geometry_df["frobenius_distance_to_R_bar"].tolist()
    )),
}
GEOMETRY_REPORT_JSON = ANALYSIS_ROOT / "seed_geometry_report.json"
GEOMETRY_REPORT_JSON.write_text(
    json.dumps(geometry_report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
for path in ANALYSIS_ROOT.iterdir():
    if path.is_file():
        shutil.copy2(path, DRIVE_ANALYSIS_ROOT / path.name)
print(f"[backup] geometry outputs -> {DRIVE_ANALYSIS_ROOT}")


**Cell 24**

## 12. Build five exact mean adapters and compute $R_{\mathrm{mean}}$

The five rank-8 updates for one attribute are concatenated into one
rank-40 adapter with unchanged `lora_alpha=16`. This exactly represents
the arithmetic mean because the new PEFT scaling becomes
$16/40=(1/5)(16/8)$.


In [ ]:
# Cell 25
from scripts.average_lora_adapters import average_lora_adapters

def publish_mean_axis(source_parent, destination_parent, axis):
    source_parent, destination_parent = Path(source_parent), Path(destination_parent)
    source_manifest = source_parent / "adapter/averaging_manifest.json"
    source_record = json.loads(source_manifest.read_text(encoding="utf-8"))
    if destination_parent.exists():
        saved_manifest = destination_parent / "adapter/averaging_manifest.json"
        if not saved_manifest.is_file():
            raise RuntimeError(f"Incomplete Drive mean adapter: {destination_parent}")
        saved_record = json.loads(saved_manifest.read_text(encoding="utf-8"))
        if saved_record.get("binding_sha256") != source_record.get("binding_sha256"):
            raise RuntimeError(f"Drive mean adapter differs for {axis}")
        return
    temporary_parent = Path(
        tempfile.mkdtemp(prefix=f".{destination_parent.name}.", dir=str(destination_parent.parent))
    )
    temporary = temporary_parent / destination_parent.name
    try:
        shutil.copytree(source_parent, temporary)
        os.replace(temporary, destination_parent)
    finally:
        shutil.rmtree(temporary_parent, ignore_errors=True)

# Restore previously verified means before asking the averaging tool to validate them.
for axis in ATTRIBUTES:
    local_parent = MEAN_ROOT / f"dpo_{axis}"
    saved_parent = DRIVE_MEAN_ROOT / f"dpo_{axis}"
    if not local_parent.exists() and saved_parent.exists():
        shutil.copytree(saved_parent, local_parent)

mean_manifests = {}
MEAN_ADAPTER_PATHS = {}
for axis in ATTRIBUTES:
    source_adapters = [
        expert_dir(seed_root(seed), axis) / "adapter"
        for seed in TRAIN_SEEDS
    ]
    output_parent = MEAN_ROOT / f"dpo_{axis}"
    output_adapter = output_parent / "adapter"
    output_parent.mkdir(parents=True, exist_ok=True)
    manifest = average_lora_adapters(
        source_adapters,
        output_adapter,
        label=axis,
        seeds=TRAIN_SEEDS,
    )
    assert manifest["output_rank"] == 40
    assert manifest["output_lora_alpha"] == 16
    assert manifest["verification_relative_frobenius_error"] <= 1e-6
    mean_manifests[axis] = manifest
    MEAN_ADAPTER_PATHS[axis] = output_adapter
    publish_mean_axis(
        output_parent, DRIVE_MEAN_ROOT / f"dpo_{axis}", axis
    )

R_mean_adapters_gram, R_mean_adapters, mean_norms = relationship_matrices(
    MEAN_ADAPTER_PATHS
)
pd.DataFrame(
    R_mean_adapters, index=ATTRIBUTES, columns=ATTRIBUTES
).to_csv(ANALYSIS_ROOT / "R_mean_adapters_cos.csv")
pd.DataFrame(
    R_mean_adapters_gram, index=ATTRIBUTES, columns=ATTRIBUTES
).to_csv(ANALYSIS_ROOT / "R_mean_adapters_gram.csv")
comparison = {
    "frobenius_R_mean_adapters_minus_R_bar": float(
        np.linalg.norm(R_mean_adapters - R_bar, ord="fro")
    ),
    "max_abs_R_mean_adapters_minus_R_bar": float(
        np.max(np.abs(R_mean_adapters - R_bar))
    ),
    "mean_adapter_norms": dict(zip(ATTRIBUTES, map(float, mean_norms))),
}
MEAN_COMPARISON_JSON = ANALYSIS_ROOT / "mean_adapter_geometry_comparison.json"
MEAN_COMPARISON_JSON.write_text(
    json.dumps(comparison, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("R_mean_adapters (geometry of the five exact mean adapters)")
display(pd.DataFrame(
    R_mean_adapters, index=ATTRIBUTES, columns=ATTRIBUTES
).round(5))
print(json.dumps(comparison, indent=2))
for path in ANALYSIS_ROOT.iterdir():
    if path.is_file():
        shutil.copy2(path, DRIVE_ANALYSIS_ROOT / path.name)
print(f"[OK] Five exact rank-40 mean adapters -> {DRIVE_MEAN_ROOT}")


**Cell 26**

## 13. Create and download the compact NB11.1 bundle

The downloadable ZIP contains the five mean adapters and all geometry
tables. The 25 individual source adapters remain persistently on Drive.


In [ ]:
# Cell 27
import zipfile
from datetime import datetime, timezone
from google.colab import files

RUN_MANIFEST = {
    "schema_version": 1,
    "notebook": "NB11.1 multi-seed TinyLlama HelpSteer2 DPO experts",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_tag": RUN_TAG,
    "base_model": BASE_MODEL,
    "base_revision": BASE_REVISION,
    "dataset": DATASET_NAME,
    "dataset_revision": DATASET_REVISION,
    "pair_seed": PAIR_SEED,
    "training_seeds": list(TRAIN_SEEDS),
    "seed_control": (
        "fixed pair subset; training seed controls initialization, "
        "data order, and other trainer randomness"
    ),
    "attribute_order": ATTRIBUTES,
    "n_training_jobs": 25,
    "n_pairs_per_attribute": MAX_PAIRS,
    "training_script_sha256_current": sha256_file(
        PROJECT_ROOT / "scripts/0_dpo_expert.py"
    ),
    "averaging_script_sha256": sha256_file(
        PROJECT_ROOT / "scripts/average_lora_adapters.py"
    ),
    "mean_operation": "exact arithmetic mean of effective LoRA updates",
    "mean_source_rank": 8,
    "mean_output_rank": 40,
    "mean_lora_alpha": 16,
    "R_seed_primary": "effective-update cosine matrix",
    "R_bar_role": "elementwise descriptive mean of five R_seed matrices",
    "R_mean_adapters_role": "geometry recomputed from five exact mean adapters",
    "source_adapter_hashes": inventory_df[
        ["seed", "attribute", "adapter_model_sha256", "source"]
    ].to_dict(orient="records"),
    "mean_adapter_hashes": {
        axis: mean_manifests[axis]["adapter_model_sha256"]
        for axis in ATTRIBUTES
    },
}
RUN_MANIFEST_PATH = ANALYSIS_ROOT / "nb11_1_run_manifest.json"
RUN_MANIFEST_PATH.write_text(
    json.dumps(RUN_MANIFEST, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
shutil.copy2(RUN_MANIFEST_PATH, DRIVE_ANALYSIS_ROOT / RUN_MANIFEST_PATH.name)

bundle_stage = Path("/tmp/nb11_1_bundle")
shutil.rmtree(bundle_stage, ignore_errors=True)
bundle_stage.mkdir(parents=True)
shutil.copytree(MEAN_ROOT, bundle_stage / "mean_adapters")
shutil.copytree(ANALYSIS_ROOT, bundle_stage / "geometry")

bundle_path = Path(
    "/content/nb11_1_tinyllama_helpsteer2_dpo_multiseed_mean_adapters.zip"
)
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in bundle_stage.rglob("*") if p.is_file()):
        archive.write(path, path.relative_to(bundle_stage).as_posix())
bundle_sha = sha256_file(bundle_path)
sha_path = Path(str(bundle_path) + ".sha256")
sha_path.write_text(f"{bundle_sha}  {bundle_path.name}\n", encoding="utf-8")
shutil.copy2(bundle_path, DRIVE_ROOT / bundle_path.name)
shutil.copy2(sha_path, DRIVE_ROOT / sha_path.name)
print(f"bundle = {bundle_path} ({bundle_path.stat().st_size / 1e6:.1f} MB)")
print(f"SHA256 = {bundle_sha}")
print(f"25 individual seed adapters: {DRIVE_TRAINED_ROOT}")
print(f"5 exact mean adapters:         {DRIVE_MEAN_ROOT}")
print(f"R matrices and reports:        {DRIVE_ANALYSIS_ROOT}")
print("\nFinal per-seed R matrices (effective-update cosine):")
for seed in TRAIN_SEEDS:
    print(f"\nR_seed{seed}")
    display(pd.DataFrame(
        R_COS_BY_SEED[seed], index=ATTRIBUTES, columns=ATTRIBUTES
    ).round(5))
print("\nR_bar")
display(pd.DataFrame(R_bar, index=ATTRIBUTES, columns=ATTRIBUTES).round(5))
print("\nR_mean_adapters")
display(pd.DataFrame(
    R_mean_adapters, index=ATTRIBUTES, columns=ATTRIBUTES
).round(5))
files.download(str(bundle_path))
files.download(str(sha_path))
